# ___Data analysis for the paper draft___
--------------------------------------

In [1]:
!python --version

Python 3.14.4


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [42]:
fred = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1") # FRED v3
meta = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Column_Definitions_2021.csv", usecols=["column_id", "name", "units"],
                                   index_col="column_id") # metadata for FRED v3 (column names, units etc.)

lookup = pd.read_csv(r"../../data/chapter2/plant_lookup.csv", low_memory=False, encoding="latin1", usecols=["genus", "apweb.family", # let's stick the the family info from APG website
                                                                                                         "order", "group"], index_col="genus").rename(mapper={"apweb.family": "family"}, axis=1) 

In [43]:
# including all the traits Luke advised

COLLABORATION_GRADIENT_TRAITS = [
    "F00679", # RD
    "F00727", # SRL
    "F00104", # RCT
]

CONSERVATION_GRADIENT_TRAITS = [
    "F00709", # RTD
    "F00261", # RN
]

TRAITS_DICT = {
    "F00709":  "RTD",
    "F00261":  "RN",
    "F00679":  "RD",
    "F00727":  "SRL",
    "F00104":  "RCT",
}

CHOSEN_ROOT_TRAITS = COLLABORATION_GRADIENT_TRAITS + CONSERVATION_GRADIENT_TRAITS

PLANT_TAXONOMY_ACCEPTED_COLUMNS = [
    "F01286", # Genus of plant according to The Plant List
    "F01287", # Species epithet of plant according to The Plant List
    "F01289", # Family of plant according to The Plant List
    "F01290"  # Order of plant.
]

BINOMINAL_NAME = ["F01286", "F01287"]
BINOMINAL_NAME_DATA_SOURCE = ["F00018", "F00019"]
ROOT_ORDER = ["F00056"]

In [44]:
meta.loc[BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE + CHOSEN_ROOT_TRAITS + ROOT_ORDER, :]

,name,units
column_id,,
F01286,Plant taxonomy_Accepted genus_TPL,NaN
F01287,Plant Taxonomy_Accepted species_TPL,NaN
F00018,Plant taxonomy_Genus_Data Source,NaN
F00019,Plant taxonomy_Species_Data source,NaN
F00679,Root diameter,mm
F00727,Specific root length (SRL),m/g
F00104,Root cortex thickness,um
F00709,Root tissue density (RTD),g/cm3
F00261,Root N content,mg/g


In [45]:
# records with missing binominal names aren't useful to us 
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].isna().mean() # dropna(subset=BINOMINAL_NAME)

F01286    0.250743
F01287    0.256478
F01289    0.243189
F01290    0.243015
F00679    0.838835
F00727    0.843627
F00104    0.993828
F00709    0.882847
F00261    0.881728
dtype: float64

In [46]:
# that many records with missing binominal names???
fred.loc[:, BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE].isna().sum()

F01286    14340
F01287    14668
F00018    13956
F00019    14612
dtype: int64

In [47]:
# thought we could use the data source's binominal names where FRED's binominal names are missing but looks like that won't help :/
# drop all the rows that do not have genus and species names

fred.dropna(subset=BINOMINAL_NAME, inplace=True)

### ___$1^{st}$ order roots___
__________________

In [48]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1")

,F01286,F01287,F01289,F01290,F00679,F00727,F00104,F00709,F00261,F00056
0,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales,NaN,NaN,NaN,NaN,NaN,1.0
3,Cunninghamia,lanceolata,Cupressaceae,Pinales,NaN,NaN,NaN,NaN,NaN,1.0
6,Magnolia,baillonii,Magnoliaceae,Magnoliales,NaN,NaN,NaN,NaN,NaN,1.0
11,Acacia,auriculiformis,Fabaceae,Fabales,NaN,NaN,NaN,NaN,NaN,1.0
15,Gordonia,axillaris,Theaceae,Ericales,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...
57129,Prunus,persica,Rosaceae,Rosales,0.146100,100.611190,NaN,NaN,NaN,1.0
57134,Prunus,persica,Rosaceae,Rosales,0.162800,45.507204,NaN,NaN,NaN,1.0
57139,Vitis,vinifera,Vitaceae,Vitales,0.130600,0.052900,NaN,NaN,NaN,1.0
57144,Vitis,vinifera,Vitaceae,Vitales,0.146067,0.035900,NaN,NaN,NaN,1.0


In [70]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
0,Dicranopteris,linearis
3,Cunninghamia,lanceolata
6,Magnolia,baillonii
11,Acacia,auriculiformis
15,Gordonia,axillaris
...,...,...
56989,Prunus,domestica
57049,Prunus,dulcis
57109,Malus,domestica
57124,Prunus,persica


In [52]:
fred.loc[:, CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").isna().mean().rename(index=TRAITS_DICT)

RD        0.276410
SRL       0.673550
RCT       0.933280
RTD       0.733519
RN        0.732327
F00056    0.000000
dtype: float64

### ___RD $\le$ 2.0 mm___
-----------------------

In [58]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000")

,F01286,F01287,F01289,F01290,F00679,F00727,F00104,F00709,F00261
54,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN,NaN
55,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN,NaN
56,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN,NaN
57,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN,NaN
58,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
57149,Vitis,vinifera,Vitaceae,Vitales,0.162800,0.0591,NaN,NaN,NaN
57150,Vitis,vinifera,Vitaceae,Vitales,0.213133,0.0214,NaN,NaN,NaN
57151,Vitis,vinifera,Vitaceae,Vitales,0.310233,0.0079,NaN,NaN,NaN
57152,Vitis,vinifera,Vitaceae,Vitales,0.600550,0.0029,NaN,NaN,NaN


In [71]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
54,Populus,tremuloides
55,Acer,negundo
56,Juglans,nigra
57,Quercus,rubra
58,Carya,glabra
...,...,...
56779,Ribes,nigrum
56989,Prunus,domestica
57049,Prunus,dulcis
57124,Prunus,persica


In [59]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").isna().mean().rename(index=TRAITS_DICT)

F01286    0.000000
F01287    0.000000
F01289    0.002301
F01290    0.002301
RD        0.000000
SRL       0.378179
RCT       0.958222
RTD       0.490918
RN        0.795955
dtype: float64

In [57]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000")

,F01286,F01287,F01289,F01290,F00679,F00727,F00104,F00709,F00261
54,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN,NaN
55,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN,NaN
56,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN,NaN
57,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN,NaN
58,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
57148,Vitis,vinifera,Vitaceae,Vitales,0.590948,0.0009,NaN,NaN,NaN
57149,Vitis,vinifera,Vitaceae,Vitales,0.162800,0.0591,NaN,NaN,NaN
57150,Vitis,vinifera,Vitaceae,Vitales,0.213133,0.0214,NaN,NaN,NaN
57151,Vitis,vinifera,Vitaceae,Vitales,0.310233,0.0079,NaN,NaN,NaN


In [73]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
54,Populus,tremuloides
55,Acer,negundo
56,Juglans,nigra
57,Quercus,rubra
58,Carya,glabra
...,...,...
56779,Ribes,nigrum
56989,Prunus,domestica
57049,Prunus,dulcis
57124,Prunus,persica


In [60]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").isna().mean().rename(index=TRAITS_DICT)

F01286    0.000000
F01287    0.000000
F01289    0.001303
F01290    0.001303
RD        0.000000
SRL       0.390120
RCT       0.957377
RTD       0.501434
RN        0.810740
dtype: float64